# 01 — Exploratory Data Analysis

This notebook explores the ENTSO-E load data and Open-Meteo weather data for Morocco.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

sns.set_theme(style='whitegrid')
ROOT = Path('..')

## 1. Load the data

In [ ]:
from src.data.fetch_data import fetch_entsoe_load, fetch_openmeteo_weather

# Fetch (or load from cache if already downloaded)
load = fetch_entsoe_load()
weather = fetch_openmeteo_weather()

print(f'Load series: {load.shape}, range: {load.index[0]} → {load.index[-1]}')
print(f'Weather df : {weather.shape}')

## 2. Load overview

In [ ]:
print(load.describe())
print(f'Missing values: {load.isna().sum()} / {len(load)}')

In [ ]:
fig = px.line(load.reset_index(), x=load.index.name or 'index', y=load.values,
              title='Morocco Hourly Electricity Demand',
              labels={'y': 'Load (MW)', 'x': 'Time'})
fig.show()

## 3. Seasonality analysis

In [ ]:
df = load.to_frame('load_MW')
df['hour'] = df.index.hour
df['dow']  = df.index.dayofweek
df['month'] = df.index.month

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

df.groupby('hour')['load_MW'].mean().plot(ax=axes[0], title='Avg Load by Hour')
axes[0].set_xlabel('Hour')

df.groupby('dow')['load_MW'].mean().plot(ax=axes[1], kind='bar', title='Avg Load by Day of Week')
axes[1].set_xticklabels(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'], rotation=0)

df.groupby('month')['load_MW'].mean().plot(ax=axes[2], kind='bar', title='Avg Load by Month')
axes[2].set_xlabel('Month')

plt.tight_layout()
plt.show()

## 4. Correlation: Load vs Weather

In [ ]:
merged = df[['load_MW']].join(weather, how='inner')
corr = merged.corr()[['load_MW']].drop('load_MW').sort_values('load_MW', ascending=False)
corr.plot(kind='barh', title='Pearson Correlation with Load', figsize=(8, 5))
plt.tight_layout()
plt.show()

## 5. Autocorrelation

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
plot_acf(load.dropna(), lags=168, ax=axes[0], title='ACF (7 days)')
plot_pacf(load.dropna(), lags=48, ax=axes[1], title='PACF (48h)')
plt.tight_layout()
plt.show()